In [1]:
import sys
import os

ROOT = os.path.abspath("..")  # adjust if needed
print(ROOT)
OPENPCDET_PATH = '/storage/project/r-gchou3-0/spanse30/OpenPCDet'
print(OPENPCDET_PATH)
# # Put OpenPCDet FIRST
sys.path.insert(0, OPENPCDET_PATH)

# # Import OpenPCDet pcdet
import pcdet
print("Using pcdet from:", pcdet.__file__)

import pickle as pkl

import os
from pathlib import Path
import copy
import numpy as np
import torch

from pcdet.config import cfg, cfg_from_yaml_file
from pcdet.models import build_network, load_data_to_gpu
from pcdet.datasets import build_dataloader
from pcdet.utils import common_utils
import random

import math
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio

import importlib
import helpers_ptt
importlib.reload(helpers_ptt)
import torch.nn.functional as F

from collections import defaultdict

from pcdet.config import cfg, cfg_from_yaml_file
cfg.clear()

CFG_FILE_MF = '/storage/project/r-gchou3-0/spanse30/OpenPCDet/tools/cfgs/waymo_models/centerpoint_4frames.yaml' 
CKPT_MF = '/storage/project/r-gchou3-0/spanse30/OpenPCDet/output/cfgs/custom_models/centerpoint_multiframe_waymo/default/ckpt/checkpoint_epoch_36.pth'  # <- put your ckpt here

logger = common_utils.create_logger()
logger.info(f'Loaded cfg from {CFG_FILE_MF}')

cfg_from_yaml_file(
    CFG_FILE_MF,
    cfg
)

cfg.TAG = 'centerpoint_multiframe'
cfg.EXP_GROUP_PATH = 'waymo_multiframe'

dataset_mf, test_loader_mf, _ = build_dataloader(
    dataset_cfg=cfg.DATA_CONFIG,
    class_names=cfg.CLASS_NAMES,
    batch_size=1,
    dist=False,
    workers=4,
    logger=logger,
    training=False
)

len_test = len(dataset_mf)
logger.info(f'Test set length: {len_test}')
cp_mf_model = build_network(
    model_cfg=cfg.MODEL,
    num_class=len(cfg.CLASS_NAMES),
    dataset=dataset_mf
)

logger.info(f'Loading checkpoint from: {CKPT_MF}')
cp_mf_model.load_params_from_file(filename=CKPT_MF, logger=logger, to_cpu=False)
cp_mf_model.cuda()
cp_mf_model.eval()

data_iter_mf = iter(test_loader_mf)
batch_dict_mf = next(data_iter_mf)

load_data_to_gpu(batch_dict_mf)
    

/storage/scratch1/9/spanse30
/storage/project/r-gchou3-0/spanse30/OpenPCDet
Using pcdet from: /storage/project/r-gchou3-0/spanse30/OpenPCDet/pcdet/__init__.py


2026-03-18 14:49:41,479   INFO  Loaded cfg from /storage/project/r-gchou3-0/spanse30/OpenPCDet/tools/cfgs/waymo_models/centerpoint_4frames.yaml
2026-03-18 14:49:41,505   INFO  Loading Waymo dataset
2026-03-18 14:49:43,249   INFO  Total skipped info 0
2026-03-18 14:49:43,249   INFO  Total samples for Waymo dataset: 39987
2026-03-18 14:49:43,250   INFO  Test set length: 39987
2026-03-18 14:49:43,495   INFO  Loading checkpoint from: /storage/project/r-gchou3-0/spanse30/OpenPCDet/output/cfgs/custom_models/centerpoint_multiframe_waymo/default/ckpt/checkpoint_epoch_36.pth
2026-03-18 14:49:43,497   INFO  ==> Loading parameters from checkpoint /storage/project/r-gchou3-0/spanse30/OpenPCDet/output/cfgs/custom_models/centerpoint_multiframe_waymo/default/ckpt/checkpoint_epoch_36.pth to GPU
2026-03-18 14:49:43,761   INFO  ==> Checkpoint trained from version: pcdet+0.6.0+233f849
2026-03-18 14:49:43,770   INFO  ==> Done (loaded 288/288)


In [2]:
from pcdet.config import cfg, cfg_from_yaml_file
cfg.clear()


CFG_FILE_SF = '/storage/project/r-gchou3-0/spanse30/OpenPCDet/tools/cfgs/waymo_models/centerpoint.yaml' 
CKPT_SF = '/storage/project/r-gchou3-0/spanse30/OpenPCDet/output/cfgs/custom_models/centerpoint_singleframe_waymo/default/ckpt/checkpoint_epoch_30.pth'  # <- put your ckpt here

cfg_from_yaml_file(CFG_FILE_SF, cfg)
cfg.TAG = Path(CFG_FILE_SF).stem
cfg.EXP_GROUP_PATH = 'centerpoint_waymo_demo'

import os
# cfg.DATA_CONFIG.DATA_PATH = os.path.join(OPENPCDET_PATH, 'data', 'waymo')

logger_sf = common_utils.create_logger()
logger_sf.info(f'Loaded cfg from {CFG_FILE_SF}')


dataset_sf, test_loader_sf, _ = build_dataloader(
    dataset_cfg=cfg.DATA_CONFIG,
    class_names=cfg.CLASS_NAMES,
    batch_size=1,
    dist=False,
    workers=4,
    logger=logger_sf,
    training=False
)

len_test_sf = len(dataset_sf)
logger.info(f'Test set length: {len_test_sf}')
model_sf = build_network(
    model_cfg=cfg.MODEL,
    num_class=len(cfg.CLASS_NAMES),
    dataset=dataset_sf
)

logger.info(f'Loading checkpoint from: {CKPT_SF}')
model_sf.load_params_from_file(filename=CKPT_SF, logger=logger, to_cpu=False)
model_sf.cuda()
model_sf.eval()

data_iter_sf = iter(test_loader_sf)
batch_dict_sf = next(data_iter_sf)

load_data_to_gpu(batch_dict_sf)

2026-03-18 14:49:45,305   INFO  Loaded cfg from /storage/project/r-gchou3-0/spanse30/OpenPCDet/tools/cfgs/waymo_models/centerpoint.yaml
2026-03-18 14:49:45,305   INFO  Loaded cfg from /storage/project/r-gchou3-0/spanse30/OpenPCDet/tools/cfgs/waymo_models/centerpoint.yaml
2026-03-18 14:49:45,312   INFO  Loading Waymo dataset
2026-03-18 14:49:45,312   INFO  Loading Waymo dataset
2026-03-18 14:49:47,267   INFO  Total skipped info 0
2026-03-18 14:49:47,267   INFO  Total skipped info 0
2026-03-18 14:49:47,268   INFO  Total samples for Waymo dataset: 38597
2026-03-18 14:49:47,268   INFO  Total samples for Waymo dataset: 38597
2026-03-18 14:49:47,269   INFO  Test set length: 38597
2026-03-18 14:49:47,269   INFO  Test set length: 38597
2026-03-18 14:49:47,323   INFO  Loading checkpoint from: /storage/project/r-gchou3-0/spanse30/OpenPCDet/output/cfgs/custom_models/centerpoint_singleframe_waymo/default/ckpt/checkpoint_epoch_30.pth
2026-03-18 14:49:47,323   INFO  Loading checkpoint from: /storage

In [3]:
import time
import time
import torch

def measure_forward_memory_and_latency(model, batch_dict, n_warmup=3, n_runs=10, label="model"):
    """
    Measures:
      - peak GPU memory during forward pass
      - average forward latency over n_runs
      - std latency over n_runs

    Assumes:
      - model is already on CUDA and in eval mode
      - batch_dict is already moved to GPU via load_data_to_gpu(batch_dict)
    """
    assert torch.cuda.is_available(), "CUDA is required for this measurement."

    # Warmup to avoid first-run CUDA/kernel initialization effects
    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model(batch_dict)
        torch.cuda.synchronize()

    latencies_ms = []
    peak_mem_bytes_runs = []

    for _ in range(n_runs):
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

        start = time.perf_counter()
        with torch.no_grad():
            _ = model(batch_dict)
        torch.cuda.synchronize()
        end = time.perf_counter()

        latencies_ms.append((end - start) * 1000.0)
        peak_mem_bytes_runs.append(torch.cuda.max_memory_allocated())

    peak_mem_bytes = max(peak_mem_bytes_runs)
    peak_mem_mb = peak_mem_bytes / (1024 ** 2)
    peak_mem_gb = peak_mem_bytes / (1024 ** 3)

    print(f"{label}")
    print(f"  Peak forward memory: {peak_mem_mb:.2f} MB ({peak_mem_gb:.3f} GB)")
    print(f"  Avg latency:         {sum(latencies_ms)/len(latencies_ms):.2f} ms")
    print(f"  Min latency:         {min(latencies_ms):.2f} ms")
    print(f"  Max latency:         {max(latencies_ms):.2f} ms")
    print(f"  Std latency:         {torch.tensor(latencies_ms).std(unbiased=False).item():.2f} ms")

    return {
        "peak_mem_bytes": peak_mem_bytes,
        "peak_mem_mb": peak_mem_mb,
        "peak_mem_gb": peak_mem_gb,
        "latencies_ms": latencies_ms,
        "avg_latency_ms": sum(latencies_ms)/len(latencies_ms),
        "min_latency_ms": min(latencies_ms),
        "max_latency_ms": max(latencies_ms),
        "std_latency_ms": torch.tensor(latencies_ms).std(unbiased=False).item(),
    }


In [4]:
# CenterPoint single-frame
sf_stats = measure_forward_memory_and_latency(
    model_sf,
    batch_dict_sf,
    n_warmup=3,
    n_runs=10,
    label="CenterPoint Single-Frame"
)

# CenterPoint multi-frame
mf_stats = measure_forward_memory_and_latency(
    cp_mf_model,
    batch_dict_mf,
    n_warmup=3,
    n_runs=10,
    label="CenterPoint Multi-Frame"
)

/storage/project/r-gchou3-0/spanse30/OpenPCDet/pcdet/ops/iou3d_nms/iou3d_nms_utils.py:66: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:83.)
  overlaps_bev = torch.cuda.FloatTensor(torch.Size((boxes_a.shape[0], boxes_b.shape[0]))).zero_()  # (N, M)


CenterPoint Single-Frame
  Peak forward memory: 521.82 MB (0.510 GB)
  Avg latency:         17.40 ms
  Min latency:         16.67 ms
  Max latency:         18.31 ms
  Std latency:         0.49 ms
CenterPoint Multi-Frame
  Peak forward memory: 776.51 MB (0.758 GB)
  Avg latency:         17.17 ms
  Min latency:         16.68 ms
  Max latency:         18.72 ms
  Std latency:         0.57 ms
